<a href="https://colab.research.google.com/github/aniget/SoftUni-AI-Integrations-for-developers/blob/main/Vector%20Databases%2C%20Embeddings%20and%20RAG/RAG_with_Anthropic_COmpare_to_OpenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG (Retrieval-Augmented Generation) with Anthropic Claude

## What is RAG?
RAG is a technique that enhances an AI model's responses by first **retrieving** relevant information from an external knowledge base, then **augmenting** the model's prompt with that information, so the model can **generate** a more accurate and grounded answer.

Without RAG, the model can only rely on its training data. With RAG, it can answer questions about **your specific data** (e.g. a course catalog, internal docs, a product database).

## How this notebook works
1. We store course descriptions in **ChromaDB** — a local vector database
2. When a user asks a question, Claude decides to call a **`lookup_course` tool**
3. The tool queries ChromaDB using **semantic similarity** (finding meaning, not just keywords)
4. The results are passed back to Claude, which uses them to form a helpful answer

This entire flow uses the **Anthropic Claude API** instead of OpenAI.

In [1]:
# Install required packages:
# - chromadb: vector database for storing and searching course descriptions
# - anthropic: the Claude API client
!pip install -q chromadb anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [2]:
import json

from pprint import pprint
from pydantic import BaseModel, Field
from typing import List

# -------------------------------------------------------------------------
# print_response() — a helper to display API responses in a readable format
#
# NOTE: Unlike OpenAI's Responses API, Anthropic's messages.create() returns
# a Message object where:
#   - response.content  → list of content blocks (TextBlock, ToolUseBlock, etc.)
#   - response.usage    → token usage (no input_tokens_details on standard calls)
#
# We iterate over content blocks and handle each type separately.
# -------------------------------------------------------------------------
def print_response(response):
    print(f"Response ID: {response.id}")
    print(f"Input Tokens: {response.usage.input_tokens} | Output Tokens: {response.usage.output_tokens}")
    print(f"Stop Reason: {response.stop_reason}")
    print()

    print(f"{'-' * 20} [Content Blocks] {'-' * 20}")
    for block in response.content:
        if block.type == "text":
            # TextBlock: the model's plain text reply
            print(f"[TEXT]\n{block.text}")
        elif block.type == "tool_use":
            # ToolUseBlock: the model wants to call one of our tools
            # block.name   → which tool to call
            # block.input  → the arguments the model chose (already a dict, not JSON string)
            print(f"[TOOL CALL] {block.name}")
            print(f"  ID:   {block.id}")
            print(f"  Args: {block.input}")

In [3]:
# -------------------------------------------------------------------------
# Course catalog — this is the knowledge base we want Claude to search.
#
# In a real application this could come from a database, an API, or files.
# Each course has:
#   - id:          unique identifier (used as the ChromaDB document ID)
#   - name:        short display name
#   - description: full text that will be embedded and searched semantically
# -------------------------------------------------------------------------
courses = [
    {
        "id": "ai_ass_dev",
        "name": "AI-Assisted Development",
        "description": "\"AI-Assisted Development\" is a practically oriented course that changes the way developers write and maintain code. In modern development, AI assistants have become a necessity for teams seeking greater speed, quality, and consistency throughout the entire code lifecycle. The course is designed for programmers who want to multiply their productivity by integrating GitHub Copilot, Cursor, Augment Code, and Claude Code directly into their daily workflow. During the training, students will master effective prompt techniques, autocompletion and refactoring with Copilot, multi-file changes and agentic flows in Cursor, test generation, documentation, and safe migrations with Augment Code, as well as large-scale edits and code review with large context in Claude Code."
    },
    {
        "id": "ai_int_dev",
        "name": "AI Integrations for Developers",
        "description": "\"AI Integrations for Developers\" is a practical course that will teach students to embed AI functionalities directly into their applications: integrations with OpenAI, Anthropic (Claude), and OpenRouter, embeddings, vector databases, and RAG for semantic search, multimodal services (images, speech, audio, video), and model fine-tuning. The emphasis is on reliability, cost, and security: streaming and structured outputs, retry/timeout/rate limits, telemetry, metrics, data protection, and fallback strategies for non-deterministic responses. Each topic includes an exercise, and the finale features a Workshop: Real-Life Project and a Workshop: Local AI (working with local models such as Ollama/vLLM). Result: confident, production-ready AI integrations."
    },
    {
        "id": "ai_ag_work_dev",
        "name": "AI Agents & Workflows for Developers",
        "description": "\"AI Agents & Workflows for Developers\" is a practical course on designing and implementing agents and orchestrations in real-world applications: n8n for visual automations and integrations, LangChain Agents & Tools for tool-augmented agents, Memory & Human-in-the-Loop for state management and approvals, and LangGraph for reliable multi-agent systems. The emphasis is on reliability, observability, and safety: routing and retries, timeouts, guardrails, logs/metrics and cost tracking, as well as HITL processes for quality. Each topic includes an exercise, followed by Exam Preparation, and the finale is a Workshop: Real-Life Project with end-to-end construction of an agent architecture. Result: robust, extensible, and controllable AI workflows in a production environment."
    },
    {
        "id": "csharp_adv",
        "name": "C# Advanced",
        "description": "The \"C# Advanced\" course builds upon the skills for working with the C# language and the .NET platform, covering more complex concepts typical of the language. During the course, students will learn to create and work with linear data structures. They will expand their knowledge of working with arrays by learning to work with multidimensional arrays or matrices. They will have the opportunity to familiarize themselves with the Generics concept — creating template classes and methods. They will solve algorithmic problems (problem-solving skills) and work with streams, files, and directories. Attention will be given to the functional programming paradigm, as well as its primary tool — LINQ for processing data streams. The development environment used by the training team will be Microsoft Visual Studio 2022, but each student is free to use their preferred tools. Additionally, 30% of the exercise tasks will be solved with the help of AI in order to encourage the use of modern technologies for process automation, while simultaneously developing skills for the effective application of AI tools in real-world conditions."
    },
    {
        "id": "csharp_oop",
        "name": "C# OOP",
        "description": "The \"C# OOP\" course will teach students the principles of object-oriented programming (OOP), how to work with classes and objects, use object-oriented modeling, and build class hierarchies. The fundamental principles of OOP will be studied, such as abstraction (interfaces and abstract classes), encapsulation, inheritance, and polymorphism. The course will delve into the most commonly used design patterns (creational, structural, and behavioral design patterns). Participants will become familiar with the SOLID principles of object-oriented software design. Various debugging techniques will be covered. Students will learn how to create and use decorators. Attention will be given to component testing (writing unit tests) and the concept of Test-Driven Development (TDD). Additionally, 30% of the exercise tasks will be solved with the help of AI in order to encourage the use of modern technologies for process automation, while simultaneously developing skills for the effective application of AI tools in real-world conditions."
    },
    {
        "id": "js_adv",
        "name": "JS Advanced",
        "description": "In the \"JS Advanced\" course, students will gain in-depth knowledge of the JavaScript language, including syntax fundamentals, working with arrays, matrices, objects, classes, and writing functions. They will study more complex concepts such as function context, explicit binding, and the event loop. The course will develop their algorithmic thinking. Upon successful completion of this course, they will be able to work with the DOM tree, perform manipulations on it, and work with events. The functional and OOP approaches to programming with JavaScript will be covered, studying concepts such as inheritance, object composition, and the prototype chain. Additionally, 30% of the exercise tasks will be solved with the help of AI in order to encourage the use of modern technologies for process automation, while simultaneously developing skills for the effective application of AI tools in real-world conditions."
    },
    {
        "id": "js_appl",
        "name": "JS Applications",
        "description": "In the \"JS Applications\" course, students will learn what HTTP requests are and how to use them. They will become familiar with REST services, what a BaaS (Backend as a Service) is and how to work with it, what asynchronous code means (Promises, using async/await), and what Templating and Routing are. During the course, they will create a Single Page Application using the techniques learned from previous lectures. They will understand the architecture of an application and how to properly structure their applications. Toward the end of the course, they will explore various design patterns and their practical applications, create their own web components using the Web Components standard, and set up a Webpack environment from scratch. Additionally, 30% of the exercise tasks will be solved with the help of AI in order to encourage the use of modern technologies for process automation, while simultaneously developing skills for the effective application of AI tools in real-world conditions."
    },
    {
        "id": "djng_basics",
        "name": "Django Basics",
        "description": "In the Django Basics course, we will lay the foundations of web programming with Python and Django. We will explore how networks actually work, what HTTP is, and what the fundamental principles of web development are. The course will cover the core concepts of the MTV (Model–Template–View) architecture, such as Function-Based Views and Class-Based Views, and in addition to these, we will use forms (Form and ModelForm) for application development, work with media files, and store data in PostgreSQL. The training includes practical exercises (labs) and workshops for building complete, fully functional Django web applications."
    },
    {
        "id": "py_adv",
        "name": "Python Advanced",
        "description": "The \"Python Advanced\" course builds upon the skills for working with the Python language, covering more complex concepts typical of the language. During the course, students will learn to create linear data structures, solve algorithmic problems (problem-solving skills), and work with stacks and queues, tuples and sets, matrices (multidimensional lists), as well as files and directories. Attention will be given to the functional programming paradigm. Recursive functions and functions with multiple arguments will be explored in greater depth. The development environment used by the training team will be PyCharm, but each student is free to use their preferred tools. Additionally, 30% of the exercise tasks will be solved with the help of AI in order to encourage the use of modern technologies for process automation, while simultaneously developing skills for the effective application of AI tools in real-world conditions."
    }
]

## Setting up ChromaDB

**ChromaDB** is a vector database — it stores text as numerical vectors (embeddings) so we can search by *meaning* rather than exact keywords.

For example, searching for *"machine learning in production"* can still find a course about *"AI integrations"* because they are semantically similar, even if the words don't match.

`PersistentClient` saves the database to disk so it survives notebook restarts.

In [4]:
from chromadb import PersistentClient

# Path where ChromaDB will save its data on the Colab VM disk
PATH_TO_CHROMA = "/content/chromadb"
chroma_client = PersistentClient(PATH_TO_CHROMA)

In [7]:
# get_or_create_collection: creates the collection if it doesn't exist yet,
# or opens the existing one if it does (safe to run multiple times).
# A "collection" is like a table — it groups related documents together.
chroma_collection = chroma_client.get_or_create_collection(name="softuni-courses")

In [8]:
# Prepare data in the format ChromaDB expects:
#   ids        → unique string identifier per document (used to avoid duplicates)
#   metadatas  → extra structured info stored alongside the document (not embedded)
#   documents  → the actual text that gets converted to embeddings and searched
#
# ChromaDB automatically generates embeddings using its built-in model.
# In production you might use a custom embedding model for better quality.
ids = [c["id"] for c in courses]
metadatas = [{"name": c["name"]} for c in courses]
documents = [c["description"] for c in courses]

# NOTE: If you run this cell twice you'll get a duplicate ID error.
# To reset, delete /content/chromadb and re-run from the PersistentClient cell.
chroma_collection.add(ids=ids, metadatas=metadatas, documents=documents)
print(f"Added {len(ids)} courses to ChromaDB")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 104MiB/s]


Added 9 courses to ChromaDB


## Setting up the Anthropic Client

We use **Google Colab Secrets** to store our API key safely (the lock icon in the left sidebar → "Secrets").
Never hardcode API keys directly in your notebook — they could be leaked if shared.

Add your key as `CLAUDE_API_KEY` in Colab Secrets.

In [9]:
from google.colab import userdata
from anthropic import Anthropic

# Retrieve the API key from Colab Secrets (set via the lock icon in the sidebar)
# Change the name of the CLAUDE api key accordingly
# First run you will have to provide access to that key
api_key = userdata.get('CLAUDEAI_API_KEY')
anthropic_client = Anthropic(api_key=api_key)

## Defining the Tool

In the Anthropic API, **tools** are functions that Claude can decide to call when it needs external information.

The flow is:
1. We send a message to Claude along with a list of available tools and their schemas
2. Claude decides! whether to answer directly or call a tool
3. If it calls a tool, we execute the function ourselves and send back the result
4. Claude uses the result to form its final answer

This is different from OpenAI's approach — Anthropic uses `input_schema` (JSON Schema format) to describe tool parameters.

In [10]:
# -------------------------------------------------------------------------
# Pydantic models for our tool
#
# LookupCourseReq defines the INPUT to the lookup_course tool.
# Pydantic validates the data and can generate a JSON schema from it.
#
# CourseInfo defines the OUTPUT — what we return to Claude after searching.
# -------------------------------------------------------------------------

class LookupCourseReq(BaseModel):
    """
    Searches the course catalog using semantic similarity to find courses
    matching the user's intent. Use this tool when a user asks about
    available courses, course content, prerequisites, or recommendations.
    Returns the top matching courses ranked by relevance.
    """
    query: str = Field(
        description="A natural language search query describing the desired course topic, "
                    "skill, or technology (e.g. 'object-oriented programming with C#', "
                    "'building web apps with Django', 'AI agents and workflows')"
    )
    top_n: int = Field(
        default=5,
        gt=1,
        lt=20,
        description="The maximum number of matching courses to return, ordered by relevance"
    )


class CourseInfo(BaseModel):
    name: str
    description: str


def lookup_course(req: LookupCourseReq) -> List[CourseInfo]:
    """
    Performs a semantic similarity search in ChromaDB.
    ChromaDB converts the query text to an embedding vector by using the same LL model which was used to embed the info about the courses and finds
    the most similar document vectors already stored in the collection.
    """
    lookup_result = chroma_collection.query(
        query_texts=[req.query],   # ChromaDB embeds this automatically
        n_results=req.top_n
    )

    # lookup_result is structured as:
    #   { "ids": [[...]], "metadatas": [[...]], "documents": [[...]], ... }
    # The outer list is for batch queries; we only send one query so we use [0]
    results_count = len(lookup_result["ids"][0])
    return [
        CourseInfo(
            name=lookup_result["metadatas"][0][i]["name"],
            description=lookup_result["documents"][0][i]
        )
        for i in range(results_count)
    ]

In [11]:
# -------------------------------------------------------------------------
# Tool schema for the Anthropic API
#
# Anthropic tools use this structure:
#   name         → identifier Claude uses when calling the tool
#   description  → tells Claude WHEN and WHY to use this tool
#   input_schema → JSON Schema describing the tool's parameters
#                  (what Claude needs to fill in when calling it)
#
# We extract the JSON schema from our Pydantic model automatically
# using .model_json_schema() — no need to write it by hand.
#
# KEY DIFFERENCE from OpenAI:
#   OpenAI wraps tools in { "type": "function", "name": ..., "parameters": ... }
#   Anthropic uses a flat structure with "input_schema" instead of "parameters"
# -------------------------------------------------------------------------
lookup_course_schema = LookupCourseReq.model_json_schema()

tools = [
    {
        "name": "lookup_course",
        "description": lookup_course_schema.get("description", "Search the course catalog"),
        "input_schema": {
            "type": "object",
            "properties": lookup_course_schema["properties"],
            "required": list(lookup_course_schema["properties"].keys())
        }
    }
]

# -------------------------------------------------------------------------
# Tool call handler registry
#
# Maps tool names → Python functions that execute them.
# When Claude calls a tool, we look up the handler by name and run it.
#
# The handler:
#   1. Wraps the raw args dict into a validated Pydantic model
#   2. Calls our lookup_course() function
#   3. Converts the result to a list of plain dicts (JSON-serializable)
# -------------------------------------------------------------------------
tool_call_handlers = {
    "lookup_course": lambda args: [c.model_dump() for c in lookup_course(LookupCourseReq(**args))]
}

## Executing Tool Calls

When Claude responds with `stop_reason = "tool_use"`, it means it wants to call one or more tools before continuing.

We need to:
1. Extract the tool call blocks from the response
2. Execute each tool locally
3. Return the results in a `tool_result` message back to Claude

**KEY DIFFERENCE from OpenAI:**
- OpenAI tool results use `{ "type": "function_call_output", "call_id": ... }`
- Anthropic tool results use a `user` message containing `{ "type": "tool_result", "tool_use_id": ... }`

In [12]:
def execute_tools(response):
    """
    Finds all tool_use blocks in the response, executes them,
    and returns a list of tool_result content blocks ready to
    be sent back to Claude in the next API call.
    """
    # Filter content blocks to only ToolUseBlocks
    # (a response can contain both text and tool_use blocks)
    tool_calls = [block for block in response.content if block.type == "tool_use"]

    results = []
    for tool_call in tool_calls:
        # block.input is already a dict (Anthropic SDK parses it for us)
        # Compare to OpenAI where arguments is a raw JSON string needing json.loads()
        args = tool_call.input

        if tool_call.name not in tool_call_handlers:
            raise Exception(f"Unknown tool: {tool_call.name}")

        # Execute the tool function
        output = tool_call_handlers[tool_call.name](args)

        # Format the result as Anthropic expects it:
        # tool_use_id links this result back to the specific tool_use block
        results.append({
            "type": "tool_result",
            "tool_use_id": tool_call.id,       # must match the tool_use block's id
            "content": json.dumps(output)       # result must be a string
        })

    return results

## Running the RAG Pipeline

Now we put it all together. The full conversation flow is:

```
User question
    ↓
Claude (decides to call lookup_course tool)
    ↓
We execute lookup_course → ChromaDB semantic search
    ↓
We send tool results back to Claude
    ↓
Claude generates final answer using retrieved course info
```

In [13]:
# -------------------------------------------------------------------------
# System prompt — sets Claude's persona and instructs it to use the tool
#
# KEY DIFFERENCE from OpenAI:
#   OpenAI supports a "developer" role in the messages array.
#   Anthropic does NOT — system instructions go in the `system` parameter
#   of messages.create(), NOT as a message with role "system" or "developer".
# -------------------------------------------------------------------------
system_prompt = (
    "You are a customer support agent for SoftUni (Software University) - "
    "a leading tech education provider offering programming courses, professional "
    "training, and career development programs for aspiring and experienced developers. "
    "Your goal is to help students and prospective students resolve their questions "
    "quickly, accurately, and with genuine care for their learning journey. "
    "Use the 'lookup_course' tool whenever a student asks about course content, "
    "recommendations, comparisons, prerequisites, or anything that requires knowledge "
    "of the course catalog. Always search before answering course-specific questions "
    "- do not rely on memory alone."
)

# The user's question — this kicks off the RAG pipeline
search_for_ai_related_course_prompt = (
    "Hello, I am trying to find an AI-related course. "
    "I want to integrate AI into my professional life as a software developer."
)

In [14]:
# -------------------------------------------------------------------------
# STEP 1: Send the user's question to Claude
#
# We provide the tools list so Claude knows it CAN call lookup_course.
# Claude will respond with stop_reason="tool_use" if it decides to search.
#
# The conversation list only holds "user" and "assistant" turns.
# System instructions go in the separate `system` parameter.
# -------------------------------------------------------------------------
conversation = [
    {"role": "user", "content": search_for_ai_related_course_prompt}
]

search_for_ai_related_course_response = anthropic_client.messages.create(
    model="claude-sonnet-4-20250514",
    system=system_prompt,         # system prompt goes here, NOT in messages
    messages=conversation,
    tools=tools,                  # make tools available to Claude
    max_tokens=1024
)

print("--- Step 1: Claude's initial response ---")
print_response(search_for_ai_related_course_response)
# Expected: stop_reason="tool_use" and a ToolUseBlock calling lookup_course

--- Step 1: Claude's initial response ---
Response ID: msg_01Acs8455FidJEZYygPogqBr
Input Tokens: 689 | Output Tokens: 120
Stop Reason: tool_use

-------------------- [Content Blocks] --------------------
[TEXT]
I'd be happy to help you find AI-related courses that can help you integrate AI into your software development career! Let me search our course catalog for relevant AI and machine learning courses.
[TOOL CALL] lookup_course
  ID:   toolu_01619V68o3Drt4sZBgAquhSm
  Args: {'query': 'AI artificial intelligence machine learning for software developers programming integration', 'top_n': 8}


In [15]:
# -------------------------------------------------------------------------
# STEP 2: Add Claude's response to the conversation history
#
# We must include Claude's tool_use message in the history before sending
# tool results. This keeps the conversation coherent.
#
# KEY DIFFERENCE from OpenAI:
#   OpenAI's Responses API lets you do: conversation.extend(response.output)
#   Anthropic: we append an "assistant" message with response.content as the value
# -------------------------------------------------------------------------
conversation.append({
    "role": "assistant",
    "content": search_for_ai_related_course_response.content  # list of content blocks
})

print("Conversation so far:")
for msg in conversation:
    print(f"  [{msg['role']}]: {type(msg['content'])}")

Conversation so far:
  [user]: <class 'str'>
  [assistant]: <class 'list'>


In [16]:
# -------------------------------------------------------------------------
# STEP 3: Execute the tool calls and add results to the conversation
#
# execute_tools() runs our lookup_course() function with the args Claude chose,
# queries ChromaDB, and formats the results as tool_result blocks.
#
# KEY DIFFERENCE from OpenAI:
#   OpenAI: tool results are appended directly as items in the messages list
#   Anthropic: tool results are wrapped in a single "user" message with
#              content = list of tool_result blocks
# -------------------------------------------------------------------------
tool_call_results = execute_tools(search_for_ai_related_course_response)

# Wrap results in a user message — this is how Anthropic expects tool results
conversation.append({
    "role": "user",
    "content": tool_call_results   # list of { type: tool_result, tool_use_id: ..., content: ... }
})

print("--- Tool call results (what ChromaDB returned) ---")
pprint(tool_call_results)

--- Tool call results (what ChromaDB returned) ---
[{'content': '[{"name": "AI-Assisted Development", "description": '
             '"\\"AI-Assisted Development\\" is a practically oriented course '
             'that changes the way developers write and maintain code. In '
             'modern development, AI assistants have become a necessity for '
             'teams seeking greater speed, quality, and consistency throughout '
             'the entire code lifecycle. The course is designed for '
             'programmers who want to multiply their productivity by '
             'integrating GitHub Copilot, Cursor, Augment Code, and Claude '
             'Code directly into their daily workflow. During the training, '
             'students will master effective prompt techniques, autocompletion '
             'and refactoring with Copilot, multi-file changes and agentic '
             'flows in Cursor, test generation, documentation, and safe '
             'migrations with Augment 

In [17]:
# -------------------------------------------------------------------------
# STEP 4: Send the tool results back to Claude to get the final answer
#
# Claude now has the retrieved course information and will use it to
# craft a helpful, grounded response to the user's original question.
#
# We send the full conversation history (user question + Claude's tool call
# + our tool results) so Claude has full context.
# -------------------------------------------------------------------------
continue_after_tool_calls_response = anthropic_client.messages.create(
    model="claude-sonnet-4-20250514",
    system=system_prompt,
    messages=conversation,
    tools=tools,
    max_tokens=1024
)

print("--- Step 4: Claude's final answer (RAG-augmented) ---")
print_response(continue_after_tool_calls_response)
# Expected: stop_reason="end_turn" and a TextBlock with the final answer

--- Step 4: Claude's final answer (RAG-augmented) ---
Response ID: msg_018MVQbGEfb8CToYWM553S3J
Input Tokens: 2412 | Output Tokens: 426
Stop Reason: end_turn

-------------------- [Content Blocks] --------------------
[TEXT]
Great! I found several excellent AI-related courses that would be perfect for integrating AI into your software development career. Here are the most relevant options:

## **Dedicated AI Courses for Developers:**

### 🤖 **AI-Assisted Development**
This is perfect for immediately boosting your productivity! You'll learn to use AI tools like GitHub Copilot, Cursor, Augment Code, and Claude Code in your daily development workflow. The course covers effective prompting, autocompletion, refactoring, test generation, and documentation.

### 🔧 **AI Integrations for Developers**
This practical course teaches you to embed AI functionalities directly into your applications. You'll work with OpenAI, Anthropic (Claude), embeddings, vector databases, RAG for semantic search, an

## Summary: OpenAI vs Anthropic Differences

| Concept | OpenAI | Anthropic |
|---|---|---|
| API method | `responses.create()` | `messages.create()` |
| System prompt | `{"role": "developer", "content": ...}` in messages | `system=` parameter |
| Tool schema key | `"parameters"` | `"input_schema"` |
| Tool type wrapper | `{"type": "function", ...}` | flat dict, no type wrapper |
| Tool args format | JSON string → needs `json.loads()` | already a dict in `block.input` |
| Tool result format | `{"type": "function_call_output", "call_id": ...}` | `{"type": "tool_result", "tool_use_id": ...}` |
| Tool result placement | Appended directly to messages | Wrapped in a `user` message |
| Response content | `response.output` | `response.content` |
| Text output | `response.output_text` | `block.text` (iterate content) |